In [1]:
import os
from typing import Optional, Tuple, Dict, List, Any, Optional, Callable

import torch
from torch import Tensor, double

from botorch.models.transforms.input import Normalize
from botorch.models import SingleTaskGP, ModelListGP
from gpytorch.mlls.sum_marginal_log_likelihood import SumMarginalLogLikelihood

device_str: str = "mps" if torch.mps.is_available() else "cpu"
device: torch.Device = torch.device(device_str)
dtype: torch.dtype = torch.float32

SMOKE_TEST: float = os.environ.get('SMOKE_TEST', 10)

In [2]:
from botorch.test_functions import Hartmann, SyntheticTestFunction

neg_hartmann6: SyntheticTestFunction = Hartmann(negate=True)

def outcome_constraint(
        X: Tensor[double],
    ) -> Tensor[double]:
    return X.sum(dim=-1) - 3

def weighted_obj(
        X: Tensor[double],
) -> Tensor[double]:
    return neg_hartmann6(X) * (outcome_constraint(X) <= 0).type_as(X)

In [3]:
NOISE_SE: float = 0.25
train_yvar: Tensor[double] = torch.tensor(NOISE_SE**2, device=device, dtype=dtype)

def generate_initial_data(
        n: int = 10
        ) -> Tuple[Tensor[double], Tensor[double], Tensor[double], Tensor[double]]:
    train_X: Tensor = torch.rand(n, 6, device=device, dtype=dtype)
    
    exact_objective: Tensor = neg_hartmann6(train_X).unsqueeze(-1)
    exact_constraint: Tensor = outcome_constraint(train_X).unsqueeze(-1)
    
    train_objective: Tensor = exact_objective + NOISE_SE * torch.randn_like(exact_objective)
    train_constraint: Tensor = exact_constraint + NOISE_SE * torch.randn_like(exact_constraint)

    best_observed_value: Tensor = weighted_obj(train_X).max().item()

    return (train_X, train_objective, train_constraint, best_observed_value)


def initialize_model(
        train_X: Tensor[double],
        
        train_objective: Tensor[double],
        train_constraint: Tensor[double],

        state_dict: Dict[str, Any] = None,
        ) -> Tuple[ModelListGP, SumMarginalLogLikelihood]:

    model_objective: SingleTaskGP = SingleTaskGP(
        train_X,
        train_objective,
        train_yvar.expand_as(train_objective),
        input_transform=Normalize(d=train_X.shape[-1])
    ).to(train_X)
    model_constraint: SingleTaskGP = SingleTaskGP(
        train_X,
        train_constraint,
        train_yvar.expand_as(train_constraint),
        input_transform=Normalize(d=train_X.shape[-1])
    ).to(train_X)

    model: ModelListGP = ModelListGP(model_objective, model_constraint)
    mll: SumMarginalLogLikelihood = SumMarginalLogLikelihood(model.likelihood, model)

    if state_dict is not None:
        model.load_state_dict(state_dict=state_dict)
    return mll, model



In [4]:
from botorch.acquisition.objective import GenericMCObjective

def objective_callable(
        Z: Tensor,
        X: Optional[Tensor] = None,
        ) -> Tensor:
    return Z[..., 0]
def constraint_callable(
        Z: Tensor
        ) -> Tensor:
    return Z[..., 1]

objective: GenericMCObjective = GenericMCObjective(objective=objective_callable)

In [5]:
from botorch.optim import optimize_acqf

bounds: Tensor = torch.tensor([[0.0] * 6, [1.0] * 6], device=device, dtype=dtype)

BATCH_SIZE:   int = 3   if not SMOKE_TEST else 2
NUM_RESTARTS: int = 10  if not SMOKE_TEST else 2
RAW_SAMPLES:  int = 512 if not SMOKE_TEST else 32

def optimize_acqf_and_get_observation(
        acq_func: Callable,
        ):
    candidates, _ = optimize_acqf(
        acq_function=acq_func,
        bounds=bounds,
        q=BATCH_SIZE,
        num_restarts=NUM_RESTARTS,
        raw_samples=RAW_SAMPLES,
        options={"batch_limit": 5, "maxiter": 200}
    )

    new_X: Tensor = candidates.detach()
    exact_objective: Tensor = neg_hartmann6(new_X).unsqueeze(-1)
    exact_constraint: Tensor = outcome_constraint(new_X).unsqueeze(-1)

    new_objective: Tensor = exact_objective + NOISE_SE * torch.randn_like(exact_objective)
    new_constraint: Tensor = exact_constraint + NOISE_SE * torch.randn_like(exact_constraint)

    return new_X, new_objective, new_constraint

def update_random_observation(
        best_random: List[Tensor],
        ) -> List[Tensor]:
    rand_X: Tensor = torch.rand(BATCH_SIZE, 6)

    next_random_best: Tensor = weighted_obj(rand_X).max().item()
    best_random.append(max(best_random[-1], next_random_best))
    
    return best_random


In [6]:
import time
import warnings

from botorch import fit_gpytorch_mll
from botorch.acquisition import (
    qLogExpectedImprovement,
    qLogNoisyExpectedImprovement,
)
from botorch.exceptions import BadInitialCandidatesWarning
from botorch.sampling.normal import SobolQMCNormalSampler


warnings.filterwarnings("ignore", category=BadInitialCandidatesWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)


N_TRIALS: int = 3 if not SMOKE_TEST else 2
N_BATCH: int = 20 if not SMOKE_TEST else 2
MC_SAMPLES: int = 256 if not SMOKE_TEST else 32

verbose: bool = False

best_observed_all_ei, best_observed_all_nei, best_random_all = [], [], []

for trial in range(1, N_TRIALS + 1):
    print(f"\nTrial {trial:<2} of {N_TRIALS} ", end="")
    best_observed_ei, best_observed_nei, best_random = [], [], []
    (
        train_x_ei,
        train_objective_ei,
        train_constraint_ei,
        best_observed_value_ei,
    ) = generate_initial_data(n=10)
    mll_ei, model_ei = initialize_model(
        train_x_ei, train_objective_ei, train_constraint_ei
    )

    train_x_nei, train_objectives_nei, train_constraints_nei = train_x_ei, train_objective_ei, train_constraint_ei
    best_observed_value_nei = best_observed_value_ei
    mll_nei, model_nei = initialize_model(train_x_nei, train_objectives_nei, train_constraints_nei)

    best_observed_ei.append(best_observed_value_ei)
    best_observed_nei.append(best_observed_value_nei)
    best_random.append(best_observed_value_ei)

    for iter in range(1, N_BATCH + 1):
        t0: float = time.monotonic()

        fit_gpytorch_mll(mll_ei)
        fit_gpytorch_mll(mll_nei)

        qmc_sampler: SobolQMCNormalSampler = SobolQMCNormalSampler(sample_shape=torch.Size([MC_SAMPLES]))

        qLogEI: qLogExpectedImprovement = qLogExpectedImprovement(
            model=model_ei,
            best_f=(train_objective_ei * (train_constraint_ei <= 0).to(train_objective_ei)).max(),
            sampler=qmc_sampler,
            objective=objective,
            constraints=[constraint_callable],
        )

        qLogNEI: qLogNoisyExpectedImprovement = qLogNoisyExpectedImprovement(
            model=model_nei,
            X_baseline=train_x_nei,
            sampler=qmc_sampler,
            objective=objective,
            constraints=[constraint_callable],
        )

        new_x_ei, new_objective_ei, new_constraint_ei = optimize_acqf_and_get_observation(qLogEI)
        new_x_nei, new_objective_nei, new_constraint_nei = optimize_acqf_and_get_observation(qLogNEI)

        train_x_ei: Tensor = torch.cat([train_x_ei, new_x_ei])
        train_objective_ei: Tensor = torch.cat([train_objective_ei, new_objective_ei])
        train_constraint_ei: Tensor = torch.cat([train_constraint_ei, new_constraint_ei])

        train_x_nei: Tensor = torch.cat([train_x_nei, new_x_nei])
        train_objective_nei: Tensor = torch.cat([train_objective_nei, new_objective_nei])
        train_constraint_nei: Tensor = torch.cat([train_constraint_nei, new_constraint_nei])

        best_random: List[Tensor] = update_random_observation(best_random)
        best_value_ei: float = weighted_obj(train_x_ei).max().item()
        best_value_nei: float = weighted_obj(train_x_nei).max().item()

        best_observed_ei.append(best_value_ei)
        best_observed_nei.append(best_value_nei)

        mll_ei, model_ei = initialize_model(
            train_x_ei,
            train_objective_ei,
            train_constraint_ei,
            model_ei.state_dict()
        )
        mll_nei, model_nei = initialize_model(
            train_x_nei,
            train_objective_nei
        )

        t1: float = time.monotonic()

        if verbose:
            print(
                f"\nBatch {iter:>2}: best_value (random, qEI, qNEI) = "
                f"({max(best_random):>4.2f}, {best_value_ei:>4.2f}, {best_value_nei:>4.2f})"
                f"time = {t1-t0:>4.2f}.",
                end="",
            )
        else:
            print(".", end="")
        
        best_observed_all_ei.append(best_observed_ei)
        best_observed_all_nei.append(best_observed_nei)
        best_random_all.append(best_random)


Trial 1  of 2 

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, mps:0 and cpu!